# DDQN

#### Imports

In [ ]:
import time
import gymnasium as gym
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Video
import seaborn as sns
import pandas as pd

# our modules
from blockblast.block_blast_3p_env import BlockBlast3PEnv
from dqn import DDQNAgent, BlockBlastCNNNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
CHECKPOINT_DIR = "checkpoints"

#### Setting up the environment and the agent

In [ ]:
env = BlockBlast3PEnv(render_mode=None)
agent = DDQNAgent(action_size=192)

In [ ]:
print(f"Number of policy_net parameters: {len(torch.nn.utils.parameters_to_vector(agent.policy_net.parameters()))}")

#### Training

In [ ]:
# training hyperparameters
num_episodes = 10000
batch_size = 64
target_update_freq = 10

# learning rate
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.9995

# save
training_name = "qqdn_v0"
project_root = Path("../").resolve() 
# to save checkpoints
checkpoint_dir = project_root / CHECKPOINT_DIR / training_name
checkpoint_dir.mkdir(parents=True, exist_ok=True)
# saving frequency: int or None
save_freq = num_episodes // 3   
# save_freq = None

In [ ]:
rewards_history = []

for episode in tqdm(range(num_episodes)):
    state, info = env.reset()
    episode_reward = 0
    done = False
    
    while not done:
        action = agent.select_action(state, epsilon)
        
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # store in replay buffer
        agent.store_transition(state, action, reward, next_state, done)
        
        # loss and update state
        loss = agent.update_model()
        state = next_state
        episode_reward += reward
        
    # update the target network
    if episode % target_update_freq == 0:
        agent.update_target_model()
        
    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_history.append(episode_reward)
    
    # saves
    if save_freq is not None and episode % save_freq == 0:
        agent.save_model(checkpoint_dir / f"{training_name}_{episode}.pth")

# final save
agent.save_model(checkpoint_dir / f"{training_name}_final.pth")

# Results

In [ ]:
history = [range(len(rewards_history)), rewards_history]

result_df = pd.DataFrame(
    np.array(history).T,
    columns=["num_episodes", "mean_final_episode_reward"],
)
result_df["agent"] = "DDQN v0"

In [ ]:
g = sns.relplot(
    x="num_episodes",
    y="mean_final_episode_reward",
    kind="line",
    hue="agent",
    estimator=None,
    data=result_df,
    height=7,
    aspect=2,
    alpha=0.5,
)

# Testing the agent

In [ ]:
env = BlockBlast3PEnv(render_mode="rgb_array")
agent = DDQNAgent(action_size=192)
agent.load_model("./../checkpoints/model_ep/model_ep_final.pth")

total_reward = 0
n = env.grid_size

obs, info = env.reset()
for step_idx in range(30):
    img = env.render()

    clear_output(wait=True)
    plt.figure(figsize=(14, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    print(f"--- Step {step_idx + 1} ---")
    print(f"Combo: {env.combo}  |  Pieces used: {env.pieces_used.tolist()}")
    print(f"Total reward so far: {total_reward:.1f}")

    # Collect valid actions: unused piece + valid placement
    while True:
        row = int(input("enter x"))-1
        col = int(input("enter y"))-1
        piece_idx = int(input("enter a piece"))-1

        if env.pieces_used[piece_idx]:
            continue
        if env._can_place(env.pieces_grids[piece_idx], row, col):    
            action = piece_idx * n * n + row * n + col
            break

    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    time.sleep(0.5)

    if terminated or truncated:
        img = env.render()
        clear_output(wait=True)
        plt.figure(figsize=(14, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
        print("\nGame Terminated.")
        break

print(f"\nFinal Reward: {total_reward:.1f}")
env.close()